In [ ]:
import json
import torch
from datasets import Dataset
from transformers import (
    GPT2Tokenizer, GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    Trainer, TrainingArguments
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

MODEL_DIR = "./gpt2domain"
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_DIR)
model = GPT2LMHeadModel.from_pretrained(MODEL_DIR).to(device)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

with open("instruct.json", "r", encoding="utf-8") as f:
    pairs = json.load(f)

def format_example(ex):
    return (
        "### Instruction:\n"
        + ex["instruction"].strip()
        + "\n### Response:\n"
        + ex["response"].strip()
        + tokenizer.eos_token
    )

formatted_texts = [format_example(ex) for ex in pairs]

print("SAMPLE FORMATTED EXAMPLE:\n")
print(formatted_texts[0][:500])

ds = Dataset.from_dict({"text": formatted_texts})

MAX_LEN = 256

def tok(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

ds = ds.map(tok, batched=True, remove_columns=["text"])
ds.set_format(type="torch", columns=["input_ids", "attention_mask"])

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args = TrainingArguments(
    output_dir="./gpt2-instruct",
    overwrite_output_dir=True,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
) #AI suggested setting report_to="none" to avoid unnecessary logging. And the generation settings were also suggested by AI to improve training efficiency.

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds,
    data_collator=data_collator,
)

trainer.train()

trainer.save_model("./gpt2-instruct-final")
tokenizer.save_pretrained("./gpt2-instruct-final")

print("Saved instruct model to ./gpt2-instruct-final")


device: cuda
SAMPLE FORMATTED EXAMPLE:

### Instruction:
What are the common symptoms of acute myocardial infarction?
### Response:
Common symptoms include chest pain or discomfort (often radiating to the arm, jaw, or back), shortness of breath, nausea, diaphoresis (sweating), and lightheadedness.<|endoftext|>


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Step,Training Loss
10,3.007200
20,1.993900
30,1.640000


Saved instruct model to ./gpt2-instruct-final


In [ ]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

MODEL_DIR = "./gpt2-instruct-final"

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_DIR)
model = GPT2LMHeadModel.from_pretrained(MODEL_DIR).to(device)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

def ask(q, max_new_tokens=120):
    prompt = f"### Instruction:\n{q.strip()}\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    out = model.generate(
    **inputs,
    max_new_tokens=80,
    do_sample=True,
    top_p=0.9,
    temperature=0.6,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id
) #AI suggested these generation settings to improve response quality.


    full = tokenizer.decode(out[0], skip_special_tokens=False)
 
    answer = full.split("### Response:\n", 1)[-1]
    print("Q:", q)
    print("A:", answer.strip())
    print("-" * 60)

ask("What is Cox proportional hazards regression?")
ask("What does hazard ratio (HR) mean?")
ask("What is the difference between incidence and prevalence?")


device: cuda
Q: What is Cox proportional hazards regression?
A: Cox proportional hazard regression measures the influence of a given outcome on the likelihood that it will occur, using an equation (X Q) where X is the probability of occurrence, and Y to be 0.05 in the case with no significant difference between the two groups. The model uses standard deviation to compare the predicted outcomes by comparing the expected values for each variable over time. For example if the risk
------------------------------------------------------------
Q: What does hazard ratio (HR) mean?
A: HR is the sum of all the risk factors for a disease, including age and sex. HR is calculated by dividing annual incidence rates by cumulative exposure to health care costs. The range ranges from 0%--100%.
### response(s):* Responses are the number that indicates how frequently a given illness is likely or expected; they include symptoms such as fever/nausea, wheez
-------------------------------------------------